
# **Python AI Masterclass - Fine Tuning**

# Contents
* [Session 1: Python Fundamentals ↗](https://colab.research.google.com/drive/1-l7fvQsZjT0Z98A8b8Z8n7HShQfPXxD9?usp=sharing)
* [Session 2: Machine learning with scikit-learn ↗](https://colab.research.google.com/drive/104FdR3_hqUPpSK-3PNF4w4v1f-NCGyew?usp=sharing)
* [Session 3: LLM Workflows with Gemma and Gemini ↗](https://colab.research.google.com/drive/1U3-efHDTm1T8o6rEpMrR0zh93aZGOwgS?usp=sharing)
* [Session 4a: Fine Tuning ↗](https://colab.research.google.com/drive/1ZlXaaAUgY9YBn4Rt57wFU_fa_F61X12t?usp=sharing)
* [Session 4b: RAG ↗](https://colab.research.google.com/drive/1ZlXaaAUgY9YBn4Rt57wFU_fa_F61X12t?usp=sharing)
* [Session 5a: Computer Vision Object Detection ↗](https://colab.research.google.com/drive/17ko1i9-nRmw9EvUxPJ2zWO_uMZqCEFUD?usp=sharing)
* [Session 5a: CV with PaliGemma ↗](https://colab.research.google.com/drive/1y8ftiUWMU8q1ZD8YmTtbrkpkB7iLfAOg?usp=sharing)
* [Session 5c: Semantic Segmentation ↗](https://colab.research.google.com/drive/112jM95PqwLTeFzEtTXiIA5fE8GNkDuha?usp=sharing)


# **Session 4a: Fine-Tuning a Large Language Model on Your Own Data**

## Learning Objectives
* Understand the concepts of pre-training and fine-tuning for Large Language Models (LLMs).

* Recognize the hardware requirements and limitations (GPU/TPU/CPU) for training.

* Prepare a custom dataset for fine-tuning.

* Fine-tune an LLM using Keras with a TPU backend.

* Fine-tune an LLM using the transformers library with a GPU backend.

* Understand the difference between full fine-tuning and Parameter-Efficient Fine-Tuning (PEFT) with LoRA.

* Evaluate the model's performance by comparing responses before and after fine-tuning.

* Compare a RAG workflow with Fine Tuning


## What is a Large Language Model (LLM)? 🤖
Think of a pre-trained LLM like a brilliant new research assistant who has read nearly the entire public internet. They have a vast general knowledge and can write essays, summarize articles, and answer questions on a huge range of topics. However, they haven't read your lab's specific protocols, your private research data, or the niche publications in your specialized field, and they make a lot of mistakes whilst trying to appear confident that they are correct (even when they are not).

This "general knowledge" comes from a process called pre-training, where the model is shown trillions of words of text and learns the patterns, grammar, and facts of human language.

## What is Fine-Tuning? 🎯
Fine-tuning is the process of taking that pre-trained model and training it for a bit longer on your own, smaller, domain-specific dataset, or teaching it a specific task. It's like giving your new research assistant a curated stack of your lab's most important papers and data. You aren't teaching them language from scratch; you're adapting their existing knowledge to your specific needs.

Through fine-tuning, the model can learn your domain's specific vocabulary, understand relationships between concepts in your field, and can adopt a specific style or format for its responses.

## Hardware Requirements 🚀
Training and fine-tuning LLMs involves billions of calculations. A standard computer processor (CPU) can handle complex, sequential, singular tasks. However, training requires thousands of simple tasks at the exact same time.

This is where Graphics Processing Units (GPUs) and Tensor Processing Units (TPUs) come in. They are specialised chips designed for massive parallel computation, making them great for deep learning. Fine-tuning even a small LLM is practically impossible without one. For this lesson, we'll use Colab's free-tier TPUs or GPUs.

## Frameworks and Architectures

### Model Architecture (The "Brains")
This is the design of the neural network itself. Examples include Gemma, GPT-2, Llama, etc. These are the pre-trained models we will adapt.

### Weights
This the "trained model" that the architechutre has learned from data. Often the "weights" and "architechure" are referred to collectively.

### Framework
These are the software libraries that provide the tools to load, manipulate, and train the models. We will use **Keras** (a user-friendly, high-level API). Other popular frameworks include **Hugging Face transformers**, **Tensorflow**, **PyTorch**, and more.

*You must use a framework that supports the model architecture you want to work with or vice versa!*

## Example 1: Fine-Tuning Gemma with Keras on a TPU
In this first example, we'll perform a full fine-tuning of Google's Gemma model. This means we will be updating all of the model's weights using our custom data. We'll use the Keras framework with a JAX backend, which is highly optimized for running on TPUs.

### Setup and Environment Configuration


In [ ]:
# Import basic libraries for file handling and data manipulation
import os
import pandas as pd

# Login to Kaggle Hub - get credentials from https://www.kaggle.com/settings
import kagglehub
kagglehub.login()

In [ ]:
# Download models and data from Kaggle
path_gemma = kagglehub.model_download("keras/gemma3/keras/gemma3_instruct_270m")
path_gpt = kagglehub.model_download("keras/gpt2/keras/gpt2_base_en")
path_data = kagglehub.dataset_download("gpreda/medquad")

In [ ]:
# Update python libraries to use TPU in a kaggle/colab notebook
# jax 0.6.2 and keras-hub 0.21.1 seem to work
!pip install -U pip -q
!pip install -U "jax[tpu]"==0.6.2 -q
!pip install keras-hub==0.21.1 -U -q

In [ ]:
# --- Environment Setup for Keras with JAX on a TPU ---

# Keras is a high-level API that can run on different backends like TensorFlow, PyTorch, or JAX.
# JAX is a high-performance library from Google that is especially efficient on TPUs.
# We explicitly tell Keras to use JAX for all its computations.
os.environ["KERAS_BACKEND"] = "jax"

# This command instructs JAX to pre-allocate all available TPU memory.
# This can prevent memory fragmentation and speed up computations, but it means this notebook
# will have exclusive use of the TPU.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

In [ ]:
# --- Import Deep Learning Libraries ---

# Import JAX and configure it to use the TPU.
import jax
jax.config.update('jax_platform_name', 'tpu')
print(f"JAX is running on {jax.devices()[0].device_kind}")

# Import our main deep learning frameworks: Keras and keras-hub (forerly keras-nlp) for LLM-specific tools.
import keras
import keras_hub

# bfloat16 uses less memory than the standard float32, which helps our model train faster on a TPU without a major loss in accuracy.
# keras.config.set_floatx("bfloat16")

## Data Loading and Preparation
An LLM needs to be trained on structured examples. For a question-answering task, this means clear pairs of "prompts" (questions) and "responses" (answers). We'll load the [medquad](https://www.kaggle.com/datasets/gpreda/medquad) dataset, which contains medical questions and answers, and then create a small, targeted subset for our task.

In [ ]:
# Load and subset the data for training
df = pd.read_csv(path_data+"/medquad.csv")
# data = df.sample(n=100, random_state=42)

# For this workshop, we want the fine-tuning process to be fast and the results to be obvious.
# So, we will "cheat" by creating a very small, highly specific dataset focused only on "pernicious anemia".
# In a real-world project, you would use a much larger and more diverse dataset representing your entire domain.
df_subset_mask = df['question'].str.contains('pernicious anemia', case=False, na=False) | \
                         df['answer'].str.contains('pernicious anemia', case=False, na=False) | \
                         df['focus_area'].str.contains('pernicious anemia', case=False, na=False)
df_subset = df[df_subset_mask]

# Preview the first few lines of the data
df_subset.head(2)

### Format the Data for the LLM
We want to train OUR model on a dataset of prompt-response pairs.
We'll write a simple function to convert our DataFrame into the dictionary format **required** by the model we choose to use.
For best results, you should format the prompt and response to match the template the model was originally trained on. This often involves special tokens like `'<start_of_turn>user'` and`'<start_of_turn>model'`. Check the [Gemma model card](https://ai.google.dev/gemma/docs/core/prompt-structure) for details.

In [ ]:
# Helper function to transform our dataframe into the required format.
def format_data(df):
    prompts = []
    responses = []
    for index, row in df.iterrows():
        question = row['question']
        response = row['answer']
        if question and response:
             # prompts.append(f"<start_of_turn>user\nInstruction:\nAnswer the following question.\nQuestion:{question}\n<end_of_turn>")
             # responses.append(f"<start_of_turn>model\nResponse:{response}\n<end_of_turn>")
            prompts.append(f"{question}")
            responses.append(f"{response}")

    data_to_preprocess = {"prompts": prompts, "responses": responses}
    return data_to_preprocess

# Apply the formatting to our data.
formatted_data = format_data(df_subset)

## Loading the Pre-Trained Model
Now, we'll load the pre-trained Gemma model. We are using a Gemma3CausalLM, which is a "Causal Language Model." This means it works by predicting the very next word (or "token") in a sequence based on the words that came before it. This is the fundamental mechanism behind text generation.

In [ ]:
# Load the Gemma3 model
# `from_preset` is a convenient Keras function to load a model with its standard configuration.
# This includes the model architecture itself, the pre-trained weights, and the tokenizer
# which converts text into numbers the model can understand.
# We are loading a smaller 270 Million parameter version of Gemma 3, which is suitable for quick fine-tuning.
print("Loading model...")
causal_lm = keras_hub.models.Gemma3CausalLM.from_preset(path_gemma)

# The .summary() method gives us a look at the model's architecture.
# Pay attention to the "Total params" and "Trainable params". In this full fine-tuning
# example, they will be the same, meaning we are updating every part of the model.
causal_lm.summary()

## Test Before Fine-Tuning (Establish a Baseline)
It's crucial to see how the model performs before we fine-tune it. This gives us a baseline to measure our improvements against. We will ask it a question about our topic and see what its general knowledge provides.

In [ ]:
# Set a prompt
prompt = "What is pernicious anemia?"

In [ ]:
print("Sending prompt to model...")

# The .generate() method takes our text prompt and produces a response.
response_raw = causal_lm.generate(prompt)

print(f"{response_raw}")

## Compile and Fine-Tune the Model
Now we need to enable our model to be modified. Then we need to "compile" the model with our training options. Then we can calll .fit() to begin fine-tuning on our data.

In [ ]:
# Enable Low-Rank Adaptation (LoRA) for parameter efficient fine-tuning.
# LoRA freezes all weights on the backbone except for specific attention layer components
causal_lm.backbone.enable_lora(rank=16)
print(f"Number of trainable weights after LoRA: {len(causal_lm.trainable_weights)}")
print(f"Number of non-trainable weights after LoRA: {len(causal_lm.non_trainable_weights)}")

In [ ]:
print("Compiling the model...")
causal_lm.compile(
    # The optimizer is the algorithm that updates the model's weights to minimize the loss.
    # Adam is a very popular and effective general-purpose optimizer.
    # The `learning_rate` is the single most important hyperparameter. It controls the size of the
    # weight updates. Too large, and the training can become unstable; too small, and it will be too slow.
    # A small learning rate like 1e-4 (0.0001) is a good starting point for fine-tuning.
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    # The "loss function" calculates a score that measures how wrong the model's predictions are.
    # The goal of training is to minimize this score. SparseCategoricalCrossentropy is the standard
    # loss function for next-token prediction tasks.
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    # Metrics are used to monitor the training process. Here, we'll track accuracy.
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()]
)
print("Done.")

In [ ]:
print("Starting fine-tuning...")
causal_lm.fit(formatted_data, epochs=10, batch_size=1) # Adjust batch_size depening on VRAM available. Adjust epoch until loss plateaus
print("Fine-tuning complete!")

## Test After Fine-Tuning
Now, we ask the exact same prompt to our newly fine-tuned model. The hope is that its answer will be ~~better, more accurate~~ closer to what we have trained the model to do.

In [ ]:
print("Testing generation from the fine-tuned model:")
response_ft = causal_lm.generate(prompt)
print(f"{response_ft}")

In [ ]:
# And compare the scope of the fine-tuned model
causal_lm.summary()

In [ ]:
#Save the model to disk!
causal_lm.save_to_preset("./my-model-ft")

# Example 2: A Quick Look at GPT-2
Here we want to highlight how the choice of a different model means we have to make different choices in our data and framework. And the absoulte bare minimum amount of code for model tuning.

In [ ]:
# Load a GPT2 backbone with pre-trained weights. NOTE the differnet keras_hub.models method!
causal_lm = keras_hub.models.CausalLM.from_preset(path_gpt)

prompt = "What is pernicious anemia?"
causal_lm.generate(prompt)

In [ ]:
# Format the data into what GPT2 model expects - different to Gemma!
def format_data_gpt2(df):
    prompts = []
    responses = []
    for index, row in df.iterrows():
        question = row['question']
        response = row['answer']
        if question and response:
             responses.append(f"{response}\n")

    return responses

formatted_data_gpt2 = format_data_gpt2(df_subset)

In [ ]:
# Just use the defaults to demonstrate how lean our model training can be! (No LORA - so full fine tuning)
causal_lm.compile()

In [ ]:
causal_lm.fit(formatted_data_gpt2, epochs=10, batch_size=7)

In [ ]:
# Try again with fine tuned model
causal_lm.generate(prompt)

# How You Can Adapt This For Your Research

The examples above use a medical question-answering dataset, but the workflow is highly adaptable.

Understand your task. Pick a model. Pick a framework.  Build your pipeline!

The key is to structure your data into what your framework/model requires, that also teaches the model how to perform the task you want.

To see an approach using the Transformers framework (instead of Keras) try *Fine Tune with Transformers (GPU accelerator required)** [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1sJviybARN36t3d_i-gfmc0zDs51WDgzz?usp=sharing) or [![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/astrobutter/gdg-ai-for-science-finetuning-transformers)

## Links:
KerasHub Documentation: https://keras.io/keras_hub/api/models/gemma3/

Good Huggingface training demo: https://www.youtube.com/watch?v=uikZs6y0qgI

Gemma 3 fine tune documentation: https://ai.google.dev/gemma/docs/core/lora_tuning

Gemma 3 on Kaggle: https://www.kaggle.com/models/google/gemma-3

GPT-2 on Kaggle: https://www.kaggle.com/models/keras/gpt2

Fine Tune Gemini: https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini-use-supervised-tuning (deprecated AI studio method: https://ai.google.dev/gemini-api/docs/model-tuning)

Huggingface Transformers documentation: https://huggingface.co/docs/transformers/en/index

Huggingface Sentence-Transformers documentation: https://huggingface.co/sentence-transformers